RetailPulse 360

Notebook 05 — Footwear Size Curves & Sales Generation

Goal: This is where everything converges. Notebooks 01-04 each produced one piece of real,
grounded signal — Rossmann's store-level temporal behavior, H&M's product/color/taxonomy/
cross-sell patterns, and real customer sentiment/sizing feedback. This notebook builds the
final missing piece (footwear size-curve logic, using documented industry-standard
distributions since no dataset gave us real size-level data) and fuses all four sources
together into the actual product catalog, SKU structure, and simulated daily sales data
for all 190 Stylo stores.

Input: store_personalities.csv, stores.csv, cities.csv (Notebook 01-02),
       color_popularity.csv, cross_sell_pairs.csv (Notebook 03),
       review_sentiment.csv, sizing_feedback.csv (Notebook 04)
Output: products.csv, skus.csv, sales.csv

In [1]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# 2. LOAD ALL PHASE 1 INPUTS
# ============================================================

BASE_PATH = "/kaggle/input/datasets/hamaz911/notebook-5-dateset/"

store_personalities = pd.read_csv(BASE_PATH + "store_personalities.csv")
stores = pd.read_csv(BASE_PATH + "stores.csv")
cities = pd.read_csv(BASE_PATH + "cities.csv")
color_popularity = pd.read_csv(BASE_PATH + "color_popularity.csv")
cross_sell_pairs = pd.read_csv(BASE_PATH + "cross_sell_pairs.csv")
review_sentiment = pd.read_csv(BASE_PATH + "review_sentiment.csv")
sizing_feedback = pd.read_csv(BASE_PATH + "sizing_feedback.csv")

print("All inputs loaded:")
for name, df in [
    ("store_personalities", store_personalities), ("stores", stores), ("cities", cities),
    ("color_popularity", color_popularity), ("cross_sell_pairs", cross_sell_pairs),
    ("review_sentiment", review_sentiment), ("sizing_feedback", sizing_feedback),
]:
    print(f"  {name}: {df.shape}")

All inputs loaded:
  store_personalities: (1115, 12)
  stores: (191, 18)
  cities: (94, 4)
  color_popularity: (178, 4)
  cross_sell_pairs: (6422, 15)
  review_sentiment: (2, 5)
  sizing_feedback: (6, 5)


In [4]:
# 3. DATA QUALITY — CROSS-FILE CONSISTENCY
# ============================================================
# Individual files were already validated in their own notebooks.
# Here we check that they're consistent WITH EACH OTHER, since
# that's the new risk once multiple sources get combined.

print("Missing values across all loaded files:")
for name, df in [
    ("store_personalities", store_personalities), ("stores", stores), ("cities", cities),
    ("color_popularity", color_popularity), ("cross_sell_pairs", cross_sell_pairs),
    ("review_sentiment", review_sentiment), ("sizing_feedback", sizing_feedback),
]:
    m = df.isna().sum()
    if m.sum() > 0:
        print(f"  {name}: {dict(m[m > 0])}")
print("(nothing printed above = no missing values in any file)")

# does every physical store's rossmann_store_id actually exist in store_personalities?
physical_stores = stores[stores["channel"] == "Physical"]
orphan_personality_refs = ~physical_stores["rossmann_store_id"].isin(store_personalities["rossmann_store_id"])
print("\nPhysical stores referencing a non-existent personality:", orphan_personality_refs.sum())

# does every city in stores.csv exist in cities.csv?
orphan_city_refs = ~stores["city"].isin(cities["city"]) & (stores["channel"] == "Physical")
print("Stores referencing a non-existent city:", orphan_city_refs.sum())

print("\nreview_sentiment content:")
print(review_sentiment)
print("\nsizing_feedback content:")
print(sizing_feedback)

Missing values across all loaded files:
  stores: {'store_size': np.int64(1), 'rossmann_store_id': np.int64(1), 'dow_mon': np.int64(1), 'dow_tue': np.int64(1), 'dow_wed': np.int64(1), 'dow_thu': np.int64(1), 'dow_fri': np.int64(1), 'dow_sat': np.int64(1), 'dow_sun': np.int64(1), 'promo_lift': np.int64(1), 'trend_pct_per_year': np.int64(1), 'volatility_cv': np.int64(1), 'holiday_lift': np.int64(1)}
(nothing printed above = no missing values in any file)

Physical stores referencing a non-existent personality: 0
Stores referencing a non-existent city: 0

review_sentiment content:
  Shoe Type  avg_sentiment  median_sentiment       std  review_count
0       Men       0.182499           0.22630  0.398109          6624
1     Women       0.168222           0.13785  0.411013          1584

sizing_feedback content:
  Shoe Type sizing_feedback  mention_count confidence  \
0       Men      runs_large             15        low   
1       Men      runs_small             33        low   
2       Men

In [5]:
# 3b. CONFIRM THE ONE MISSING ROW IS STY-ECOM (EXPECTED)
# ============================================================
missing_personality_row = stores[stores["rossmann_store_id"].isna()]
print(missing_personality_row[["store_id", "channel"]])
assert missing_personality_row["store_id"].iloc[0] == "STY-ECOM", "Unexpected row missing a personality"
print("\nConfirmed: the only missing personality is the online channel, as designed.")

     store_id channel
190  STY-ECOM  Online

Confirmed: the only missing personality is the online channel, as designed.


In [6]:
# 4. DEFINE FOOTWEAR SIZE RANGES (SOURCED)
# ============================================================
# Based on documented Pakistani retail sizing conventions (EU standard,
# most commonly used by local footwear retailers):
#   Men's:   EU 39-46, most popular range 40-44, single peak ~41
#   Women's: EU 35-42, most popular range 36-40, single peak ~37
#   Kids':   EU 17-35 (age-banded in real retail, simplified here to
#            one continuous range since Stylo's SKU granularity is by
#            size, not customer age)
# Not derived from any dataset — documented industry/market convention.

SIZE_RANGES = {
    "Men's":   {"sizes": list(range(39, 47)), "peak": 41},
    "Women's": {"sizes": list(range(35, 43)), "peak": 37},
    "Kids":    {"sizes": list(range(17, 36)), "peak": 26},
}

for gender, info in SIZE_RANGES.items():
    print(f"{gender}: sizes {info['sizes'][0]}-{info['sizes'][-1]}, peak at {info['peak']}")

Men's: sizes 39-46, peak at 41
Women's: sizes 35-42, peak at 37
Kids: sizes 17-35, peak at 26


In [9]:
# 4b. KIDS SIZE CURVE — AGE-BANDED MIXTURE (MORE REALISTIC)
# ============================================================
# Single bell curve underrepresents infant/near-teen sizes. Real
# retail kids sizing is closer to several overlapping peaks by age
# band. Age bands and their typical EU size ranges/peaks are a
# documented retail convention (Khazanay, other Pakistani retailers):
#   Infant  (0-2 yrs):  EU 17-20, peak ~19
#   Toddler (2-6 yrs):  EU 21-27, peak ~24
#   Child   (6-10 yrs): EU 28-33, peak ~30
#   Teen    (10-12+):   EU 32-35, peak ~34
# Equal weighting assumed across bands (no real data on relative
# population/sales share per age group) — documented assumption.

KIDS_AGE_BANDS = {
    "Infant":  {"peak": 19, "std": 1.0},
    "Toddler": {"peak": 24, "std": 1.8},
    "Child":   {"peak": 30, "std": 1.8},
    "Teen":    {"peak": 34, "std": 1.0},
}
KIDS_SIZES = list(range(17, 36))
BAND_WEIGHT = 1 / len(KIDS_AGE_BANDS)  # equal weighting, documented assumption

def build_kids_size_curve(sizes, bands, band_weight):
    combined = np.zeros(len(sizes))
    for band, info in bands.items():
        combined += band_weight * norm.pdf(sizes, loc=info["peak"], scale=info["std"])
    combined = combined / combined.sum()
    return dict(zip(sizes, combined))

kids_curve = build_kids_size_curve(KIDS_SIZES, KIDS_AGE_BANDS, BAND_WEIGHT)
size_curves["Kids"] = kids_curve  # overwrite the earlier single-peak version

print(f"Kids size curve (sums to {sum(kids_curve.values()):.4f}):")
for size, weight in kids_curve.items():
    print(f"  Size {size}: {weight*100:.1f}%")

Kids size curve (sums to 1.0000):
  Size 17: 1.4%
  Size 18: 6.2%
  Size 19: 10.3%
  Size 20: 6.6%
  Size 21: 2.8%
  Size 22: 3.2%
  Size 23: 4.8%
  Size 24: 5.7%
  Size 25: 4.9%
  Size 26: 3.5%
  Size 27: 2.8%
  Size 28: 3.5%
  Size 29: 4.9%
  Size 30: 5.7%
  Size 31: 4.9%
  Size 32: 4.4%
  Size 33: 7.6%
  Size 34: 10.6%
  Size 35: 6.3%


In [11]:
# 5. GENERATE SIZE CURVE WEIGHTS (NORMAL DISTRIBUTION)
# ============================================================
from scipy.stats import norm

SIZE_CURVE_STD = 1.8  # controls how quickly popularity tapers from the peak size

def build_size_curve(sizes, peak, std=SIZE_CURVE_STD):
    weights = norm.pdf(sizes, loc=peak, scale=std)
    weights = weights / weights.sum()  # normalize so they sum to 1 (proportions)
    return dict(zip(sizes, weights))

size_curves = {
    gender: build_size_curve(info["sizes"], info["peak"])
    for gender, info in SIZE_RANGES.items()
}

for gender, curve in size_curves.items():
    print(f"\n{gender} size curve:")
    for size, weight in curve.items():
        print(f"  Size {size}: {weight*100:.1f}%")


Men's size curve:
  Size 39: 13.0%
  Size 40: 20.7%
  Size 41: 24.1%
  Size 42: 20.7%
  Size 43: 13.0%
  Size 44: 6.0%
  Size 45: 2.0%
  Size 46: 0.5%

Women's size curve:
  Size 35: 13.0%
  Size 36: 20.7%
  Size 37: 24.1%
  Size 38: 20.7%
  Size 39: 13.0%
  Size 40: 6.0%
  Size 41: 2.0%
  Size 42: 0.5%

Kids size curve:
  Size 17: 0.0%
  Size 18: 0.0%
  Size 19: 0.0%
  Size 20: 0.1%
  Size 21: 0.5%
  Size 22: 1.9%
  Size 23: 5.5%
  Size 24: 12.0%
  Size 25: 19.0%
  Size 26: 22.2%
  Size 27: 19.0%
  Size 28: 12.0%
  Size 29: 5.5%
  Size 30: 1.9%
  Size 31: 0.5%
  Size 32: 0.1%
  Size 33: 0.0%
  Size 34: 0.0%
  Size 35: 0.0%


In [13]:
# 6. BUILD THE PRODUCT CATALOG
# ============================================================
from itertools import product

GENDER_STYLES = ["Men's", "Women's", "Kids"]
CATEGORY_STYLES = ["Casual", "Formal", "Sports"]
STYLES_PER_COMBO = 8  # distinct product styles per Gender x Category combo

# Colors: use real H&M-derived popularity where we have signal,
# documented industry-standard defaults for the gaps (Sports, Men's/Formal)
INDUSTRY_DEFAULT_COLORS = {
    "Sports": ["Black", "White", "Grey", "Red"],
    "Men's_Formal": ["Black", "Brown", "Dark Brown"],
}

def get_top_colors(gender, style, n=4):
    subset = color_popularity[
        (color_popularity["stylo_gender"] == gender) & (color_popularity["stylo_style"] == style)
    ]
    if len(subset) == 0:
        # use documented industry default for known gaps
        if style == "Sports":
            return INDUSTRY_DEFAULT_COLORS["Sports"]
        if gender == "Men's" and style == "Formal":
            return INDUSTRY_DEFAULT_COLORS["Men's_Formal"]
        return ["Black", "White"]  # generic fallback, shouldn't be hit given our known gaps
    top = subset.sort_values("article_count", ascending=False).head(n)
    return top["colour_group_name"].tolist()

products = []
product_id_counter = 1
for gender, style in product(GENDER_STYLES, CATEGORY_STYLES):
    colors = get_top_colors(gender, style)
    for i in range(STYLES_PER_COMBO):
        products.append({
            "product_id": f"PROD-{product_id_counter:04d}",
            "gender": gender,
            "category": style,
            "style_name": f"{gender} {style} Style {i+1}",
            "available_colors": colors,
            "has_real_color_signal": len(color_popularity[
                (color_popularity["stylo_gender"] == gender) & (color_popularity["stylo_style"] == style)
            ]) > 0,
        })
        product_id_counter += 1

products_df = pd.DataFrame(products)
print("Total products (styles):", len(products_df))
print("\nProducts per Gender x Category:")
print(products_df.groupby(["gender", "category"]).size())
print("\nColor signal source:")
print(products_df.groupby(["gender","category"])["has_real_color_signal"].first())

Total products (styles): 72

Products per Gender x Category:
gender   category
Kids     Casual      8
         Formal      8
         Sports      8
Men's    Casual      8
         Formal      8
         Sports      8
Women's  Casual      8
         Formal      8
         Sports      8
dtype: int64

Color signal source:
gender   category
Kids     Casual       True
         Formal       True
         Sports      False
Men's    Casual       True
         Formal      False
         Sports      False
Women's  Casual       True
         Formal       True
         Sports      False
Name: has_real_color_signal, dtype: bool


In [14]:
# 7. EXPLODE PRODUCTS INTO SIZE x COLOR SKUs
# ============================================================

sku_rows = []
sku_counter = 1

for _, prod in products_df.iterrows():
    size_curve = size_curves[prod["gender"]]  # dict of {size: weight}
    for size, size_weight in size_curve.items():
        for color in prod["available_colors"]:
            sku_rows.append({
                "sku_id": f"SKU-{sku_counter:05d}",
                "product_id": prod["product_id"],
                "gender": prod["gender"],
                "category": prod["category"],
                "style_name": prod["style_name"],
                "size": size,
                "size_weight": size_weight,  # relative demand weight from the size curve
                "color": color,
                "has_real_color_signal": prod["has_real_color_signal"],
            })
            sku_counter += 1

skus_df = pd.DataFrame(sku_rows)

print("Total SKUs generated:", len(skus_df))
print("\nSKU count by gender:")
print(skus_df["gender"].value_counts())

# sanity check: every SKU's size_weight should sum to 1.0 per (product_id, color) group... 
# actually it sums to 1.0 per (product_id) across all sizes for a given color, since each
# color gets the full size range
check = skus_df.groupby(["product_id", "color"])["size_weight"].sum()
print("\nSize weights sum check (should all be ~1.0):")
print(check.describe())

Total SKUs generated: 3296

SKU count by gender:
gender
Kids       1824
Women's     768
Men's       704
Name: count, dtype: int64

Size weights sum check (should all be ~1.0):
count    2.800000e+02
mean     1.000000e+00
std      1.112211e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: size_weight, dtype: float64


In [15]:
# 8. ASSIGN STORE ASSORTMENTS
# ============================================================
# Large stores carry more of the catalog; Small stores carry less.
# This directly encodes the "assortment mismatch" business problem
# from Phase 0 — not every store should have every SKU.

ASSORTMENT_FRACTION = {
    "Large": 0.65,
    "Medium": 0.40,
    "Small": 0.22,
}

np.random.seed(11)

physical_stores = stores[stores["channel"] == "Physical"].copy()
all_sku_ids = skus_df["sku_id"].values

assortment_rows = []
for _, store in physical_stores.iterrows():
    frac = ASSORTMENT_FRACTION[store["store_size"]]
    n_skus = int(len(all_sku_ids) * frac)
    assigned_skus = np.random.choice(all_sku_ids, size=n_skus, replace=False)
    for sku in assigned_skus:
        assortment_rows.append({"store_id": store["store_id"], "sku_id": sku})

assortment_df = pd.DataFrame(assortment_rows)

print("Total store-SKU assortment rows:", len(assortment_df))
print("\nAvg SKUs per store, by size:")
avg_by_size = assortment_df.merge(stores[["store_id","store_size"]], on="store_id").groupby("store_size").size() / physical_stores["store_size"].value_counts()
print(avg_by_size)

Total store-SKU assortment rows: 228724

Avg SKUs per store, by size:
store_size
Large     2142.0
Medium    1318.0
Small      725.0
dtype: float64


In [16]:
# 9. BUILD THE STYLO SALES CALENDAR
# ============================================================
# Real, sourced Eid dates for our 2024-2026 range (lunar calendar,
# shifts yearly, cannot be estimated/interpolated reliably).
# Wedding season (roughly Nov-Feb, cooler months, traditional Pakistani
# wedding season) and winter (Nov-Feb, boot/closed-shoe demand) are
# well-documented general seasonal patterns, not exact dates.

import pandas as pd
from datetime import date, timedelta

START_DATE = date(2024, 2, 23)
END_DATE = date(2026, 8, 23)

EID_DATES = [
    date(2024, 4, 10),  # Eid-ul-Fitr 2024
    date(2024, 6, 17),  # Eid-ul-Adha 2024
    date(2025, 3, 31),  # Eid-ul-Fitr 2025
    date(2025, 6, 7),   # Eid-ul-Adha 2025
    date(2026, 3, 20),  # Eid-ul-Fitr 2026
    date(2026, 5, 27),  # Eid-ul-Adha 2026
]
PRE_EID_SHOPPING_DAYS = 10  # shopping window BEFORE each Eid, not on it

calendar_df = pd.DataFrame({"date": pd.date_range(START_DATE, END_DATE, freq="D")})
calendar_df["day_of_week"] = calendar_df["date"].dt.dayofweek + 1  # match Rossmann's Mon=1..Sun=7
calendar_df["month"] = calendar_df["date"].dt.month

calendar_df["is_pre_eid"] = False
for eid_date in EID_DATES:
    window_start = pd.Timestamp(eid_date) - pd.Timedelta(days=PRE_EID_SHOPPING_DAYS)
    window_end = pd.Timestamp(eid_date)
    mask = (calendar_df["date"] >= window_start) & (calendar_df["date"] < window_end)
    calendar_df.loc[mask, "is_pre_eid"] = True

calendar_df["is_wedding_season"] = calendar_df["month"].isin([11, 12, 1, 2])
calendar_df["is_winter"] = calendar_df["month"].isin([11, 12, 1, 2])

print("Calendar shape:", calendar_df.shape)
print("Pre-Eid shopping days:", calendar_df["is_pre_eid"].sum())
print("Wedding season days:", calendar_df["is_wedding_season"].sum())
calendar_df.head()

Calendar shape: (913, 6)
Pre-Eid shopping days: 60
Wedding season days: 247


,date,day_of_week,month,is_pre_eid,is_wedding_season,is_winter
0,2024-02-23,5,2,False,True,True
1,2024-02-24,6,2,False,True,True
2,2024-02-25,7,2,False,True,True
3,2024-02-26,1,2,False,True,True
4,2024-02-27,2,2,False,True,True


In [28]:
# 10. GENERATE SPARSE DAILY SALES (RECALIBRATED)
# ============================================================
# Original BASE_RATE produced ~577 units/day for a Large store —
# unrealistic, supermarket-scale for footwear retail. Recalibrated
# to a defensible target: Large ~40/day, Medium ~15/day, Small ~6/day.

BASE_RATE = {"Large": 0.15 * (40/577.0), "Medium": 0.10 * (15/115.9), "Small": 0.06 * (6/56.4)}
print("Recalibrated BASE_RATE:", BASE_RATE)

gen_df["base_rate"] = gen_df["store_size"].map(BASE_RATE)

DOW_COLS = ["dow_mon","dow_tue","dow_wed","dow_thu","dow_fri","dow_sat","dow_sun"]
np.random.seed(42)

sales_chunks = []
for day_idx, row in calendar_df.iterrows():
    d = row["date"]
    dow_col = DOW_COLS[row["day_of_week"] - 1]
    dow_ratio = gen_df[dow_col].values

    seasonal_mult = np.ones(len(gen_df))
    if row["is_pre_eid"]:
        seasonal_mult *= SEASONAL["pre_eid_all"]
    if row["is_wedding_season"]:
        seasonal_mult *= np.where(gen_df["category"] == "Formal", SEASONAL["wedding_formal"], 1.0)
    if row["is_winter"]:
        seasonal_mult *= SEASONAL["winter_all"]

    trend_factor = 1 + (gen_df["trend_pct_per_year"].values * (day_idx / 365))

    lam = (gen_df["base_rate"].values * dow_ratio * seasonal_mult *
           gen_df["size_weight_relative"].values * gen_df["color_weight_relative"].values *
           trend_factor)
    lam = np.clip(lam, 0, None)

    units = np.random.poisson(lam)
    nonzero = units > 0
    if nonzero.sum() > 0:
        chunk = pd.DataFrame({
            "date": d,
            "store_id": gen_df.loc[nonzero, "store_id"].values,
            "sku_id": gen_df.loc[nonzero, "sku_id"].values,
            "units_sold": units[nonzero],
        })
        sales_chunks.append(chunk)

sales_df = pd.concat(sales_chunks, ignore_index=True)
print("\nTotal sales rows generated:", len(sales_df))
print("Total units sold:", sales_df["units_sold"].sum())

Recalibrated BASE_RATE: {'Large': 0.010398613518197573, 'Medium': 0.012942191544434857, 'Small': 0.006382978723404255}

Total sales rows generated: 2459464
Total units sold: 2502886


In [29]:
# 11. DATA QUALITY — VALIDATE RECALIBRATED SALES
# ============================================================

daily_store_totals = sales_df.groupby(["date", "store_id"])["units_sold"].sum()
print("Per-store daily units:")
print(daily_store_totals.describe())

size_check = sales_df.merge(stores[["store_id","store_size"]], on="store_id").groupby("store_size")["units_sold"].sum() / \
             sales_df.merge(stores[["store_id","store_size"]], on="store_id").groupby("store_size")["date"].nunique() / \
             stores[stores["channel"]=="Physical"]["store_size"].value_counts()
print("\nPer-store avg daily units by size (target: Large~40, Medium~15, Small~6):")
print(size_check)

print("\nReferential integrity — orphan store_ids:", (~sales_df["store_id"].isin(stores["store_id"])).sum())
print("Referential integrity — orphan sku_ids:", (~sales_df["sku_id"].isin(skus_df["sku_id"])).sum())

Per-store daily units:
count    172808.000000
mean         14.483623
std          10.076569
min           1.000000
25%           5.000000
50%          13.000000
75%          22.000000
max          78.000000
Name: units_sold, dtype: float64

Per-store avg daily units by size (target: Large~40, Medium~15, Small~6):
store_size
Large     25.940440
Medium    20.099267
Small      5.399296
dtype: float64

Referential integrity — orphan store_ids: 0
Referential integrity — orphan sku_ids: 0


In [30]:
# 12. SAVE OUTPUTS
# ============================================================

products_df.to_csv("products.csv", index=False)
skus_df.to_csv("skus.csv", index=False)
sales_df.to_csv("sales.csv", index=False)

print("Saved products.csv —", products_df.shape)
print("Saved skus.csv —", skus_df.shape)
print("Saved sales.csv —", sales_df.shape)

Saved products.csv — (72, 6)
Saved skus.csv — (3296, 12)
Saved sales.csv — (2459464, 4)


In [31]:
# 13. NOTEBOOK SUMMARY
# ============================================================

print("NOTEBOOK 05 SUMMARY")
print(f"Product styles: {len(products_df)}")
print(f"SKUs (size x color): {len(skus_df)}")
print(f"Store-SKU assortment pairs: {len(assortment_df)}")
print(f"Sales rows (sparse, non-zero only): {len(sales_df)}")
print(f"Total units sold: {sales_df['units_sold'].sum()}")
print(f"Date range: {sales_df['date'].min()} to {sales_df['date'].max()}")
print(f"Documented gaps using industry-standard assumptions: Sports colors, Men's/Formal colors, base_rate calibration")
print("Output files: products.csv, skus.csv, sales.csv")
print("\n✓ Notebook 05 completed successfully.")

NOTEBOOK 05 SUMMARY
Product styles: 72
SKUs (size x color): 3296
Store-SKU assortment pairs: 228724
Sales rows (sparse, non-zero only): 2459464
Total units sold: 2502886
Date range: 2024-02-23 00:00:00 to 2026-08-23 00:00:00
Documented gaps using industry-standard assumptions: Sports colors, Men's/Formal colors, base_rate calibration
Output files: products.csv, skus.csv, sales.csv

✓ Notebook 05 completed successfully.


In [32]:
# 14. ZIP ARTIFACTS FOR LOCAL DOWNLOAD
# ============================================================
import shutil
from pathlib import Path

OUTPUT_FILES = [Path("products.csv"), Path("skus.csv"), Path("sales.csv")]
ZIP_NAME = "notebook_05_catalog_sales_artifact"
ARTIFACT_DIR = Path("notebook_05_outputs")

missing = [f for f in OUTPUT_FILES if not f.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing output file(s): {missing}. Run the save cell (Section 12) first."
    )

ARTIFACT_DIR.mkdir(exist_ok=True)
for f in OUTPUT_FILES:
    shutil.copy(f, ARTIFACT_DIR / f.name)

zip_path = shutil.make_archive(ZIP_NAME, "zip", root_dir=".", base_dir=ARTIFACT_DIR.name)

print("ZIP created successfully:")
print(zip_path)
print(f"ZIP size: {Path(zip_path).stat().st_size / 1024:.2f} KB")
print(f"Contents: {[f.name for f in OUTPUT_FILES]}")

ZIP created successfully:
/kaggle/working/notebook_05_catalog_sales_artifact.zip
ZIP size: 10085.24 KB
Contents: ['products.csv', 'skus.csv', 'sales.csv']
